# Cognee activation validation

Reproducible checks for the July 23, 2026 activation decision. The notebook reads only saved synthetic-benchmark evidence and does not access the Obsidian vault.

In [1]:
from pathlib import Path
import json
import math

root = Path.cwd()
evidence_dir = root / "REPORTS" / "technology-porting"
benchmark = json.loads((evidence_dir / "cognee-activation-lite-benchmark-2026-07-23.json").read_text())
prepare = json.loads((evidence_dir / "cognee-activation-lite-prepare-2026-07-23.json").read_text())
decision = json.loads((evidence_dir / "cognee-activation-decision-2026-07-23.json").read_text())
len(benchmark["details"]), prepare["documentCount"], decision["decision"]["activateNow"]

(120, 12, False)

## Input-quality and retrieval recomputation

The expected document is evaluated at ranks 1, 3, and 10 for each of 120 unique labeled queries. The token baseline is the current nova-use-style lexical recall comparator encoded in the benchmark.

In [2]:
def retrieval_metrics(details, field):
    ranks = []
    for row in details:
        try:
            ranks.append(row[field].index(row["expectedRelId"]) + 1)
        except ValueError:
            ranks.append(None)
    count = len(ranks)
    return {
        "recallAt1": round(sum(rank == 1 for rank in ranks) / count, 6),
        "recallAt3": round(sum(rank is not None and rank <= 3 for rank in ranks) / count, 6),
        "recallAt10": round(sum(rank is not None and rank <= 10 for rank in ranks) / count, 6),
        "mrr": round(sum(0 if rank is None else 1 / rank for rank in ranks) / count, 6),
    }

details = benchmark["details"]
recomputed_local = retrieval_metrics(details, "localTokenRecall")
recomputed_cognee = retrieval_metrics(details, "cognee")
assert recomputed_local == benchmark["retrieval"]["localTokenRecall"]
assert recomputed_cognee == benchmark["retrieval"]["cognee"]
quality_gain_pp = round((recomputed_cognee["recallAt3"] - recomputed_local["recallAt3"]) * 100, 2)
assert quality_gain_pp == benchmark["retrieval"]["qualityGainPercentagePointsAt3"]
{"local": recomputed_local, "cognee": recomputed_cognee, "quality_gain_pp_at_3": quality_gain_pp}

{'local': {'recallAt1': 0.766667,
  'recallAt3': 0.966667,
  'recallAt10': 1.0,
  'mrr': 0.860694},
 'cognee': {'recallAt1': 0.883333,
  'recallAt3': 0.925,
  'recallAt10': 0.983333,
  'mrr': 0.911908},
 'quality_gain_pp_at_3': -4.17}

## Latency, cache, and reliability recomputation

The percentile implementation matches the benchmark's nearest-rank definition. Cache-miss and cache-hit samples each contain 120 sequential requests.

In [3]:
def percentile(values, probability):
    ordered = sorted(values)
    index = max(0, min(len(ordered) - 1, math.ceil(probability * len(ordered)) - 1))
    return ordered[index]

miss = [row["missLatencyMs"] for row in details]
hit = [row["cacheLatencyMs"] for row in details]
assert round(percentile(miss, 0.95), 2) == benchmark["latencyMs"]["cacheMiss"]["p95"]
assert round(percentile(hit, 0.95), 2) == benchmark["latencyMs"]["cacheHit"]["p95"]
assert sum(row["missStatus"] != 200 for row in details) == 0
assert sum(row["cacheStatus"] != 200 for row in details) == 0
assert all(row["cacheResultsMatch"] for row in details)
{
    "cache_miss_p95_ms": round(percentile(miss, 0.95), 2),
    "cache_hit_p95_ms": round(percentile(hit, 0.95), 2),
    "miss_to_target_ratio": round(percentile(miss, 0.95) / 500, 2),
    "errors": 0,
}

{'cache_miss_p95_ms': 2964.4,
 'cache_hit_p95_ms': 0.73,
 'miss_to_target_ratio': 5.93,
 'errors': 0}

## Gate decision

Activation is fail-closed: every required performance, quality, safety, and recovery gate must pass. The full pipeline timeout is a separate hard failure and cannot be replaced by the chunk-only result.

In [4]:
gates = benchmark["gates"]
failed_gates = sorted(name for name, passed in gates.items() if not passed)
assert failed_gates == ["cacheMissP95AtOrBelow500Ms", "qualityGainAtLeast10Points"]
assert decision["fullCogneePipeline"]["gatePassed"] is False
activate = decision["fullCogneePipeline"]["gatePassed"] and all(gates.values())
assert activate is False
{
    "activate": activate,
    "failed_partial_candidate_gates": failed_gates,
    "full_pipeline_status": decision["fullCogneePipeline"]["cognify"]["status"],
    "automatic_shadow": decision["decision"]["automaticShadow"],
    "production_augment": decision["decision"]["productionAugment"],
    "full_migration": decision["decision"]["fullMigration"],
}

{'activate': False,
 'failed_partial_candidate_gates': ['cacheMissP95AtOrBelow500Ms',
  'qualityGainAtLeast10Points'],
 'full_pipeline_status': 'TIMED_OUT_AND_INTERRUPTED',
 'automatic_shadow': 'REJECTED',
 'production_augment': 'REJECTED',
 'full_migration': 'REJECTED'}